# WiDR Full Pipeline — Wi-Fi Dual-task Recognition

**논문**: "Wi-Fi CSI-Based Human Activity Recognition and Indoor Localization with Sampling Irregularity Mitigation" (Lee & Toh, IEEE IoT Journal 2025)

## 전체 파이프라인 개요

| 단계 | 설명 | 논문 섹션 |
|------|------|-----------|
| **Step 0** | Raw CSI 데이터 로드 및 탐색 | - |
| **Step 1** | 진폭(Amplitude) 추출 — I/Q → √(R²+I²) | - |
| **Step 2** | Null Subcarrier 제거 (192→114, HT-LTF) | - |
| **Step 3** | 872-Frame 정렬 (4 RX 동기화) | - |
| **Step 4** | 보간 (Linear Interpolation) | - |
| **Step 5** | Neural CDE — 불규칙 샘플링 완화 | Section III-A |
| **Step 6** | StandardScaler 정규화 | - |
| **Step 7** | Dual-Stream Cross-Attention 모델 | Section III-B |
| **Step 8** | Parameter-Shared Dual-Task 학습 (Mixup) | Section III-C |
| **Step 9** | 평가 및 시각화 | Section IV |

**데이터 흐름**: `13_raw_data/` → 진폭 추출 → Null 제거 → 872 정렬 → 보간 → (Neural CDE) → 정규화 → WiDRNet → Action + Zone 예측

## Step 0. 환경 설정 & 상수 정의

In [8]:
import os
import glob
import re
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torch.nn.utils import clip_grad_norm_
from torch.optim.lr_scheduler import CosineAnnealingLR
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import classification_report, confusion_matrix
from tqdm.notebook import tqdm
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

# ============================================================
# 상수 정의
# ============================================================
# 서브캐리어
NULL_IN_HT_LTF = [64, 65] + list(range(123, 134)) + [191]  # 14개 null
VALID_HT_LTF_IDX = [i for i in range(64, 192) if i not in NULL_IN_HT_LTF]
NUM_SUBCARRIERS = len(VALID_HT_LTF_IDX)  # 114

# 시퀀스
SEQ_LEN = 872
SKIP_FIRST = 50   # 초기 과도 상태 패킷 스킵
NUM_RX = 4
FEATURE_DIM = NUM_SUBCARRIERS * NUM_RX  # 456

# 레이블
ACTION_MAP = {'handsup': 0, 'sit': 1, 'stand': 2, 'walk': 3}
NUM_ACTIONS = len(ACTION_MAP)

ZONE_MAP = {
    1: 0, 2: 0, 5: 0, 6: 0,     # Zone 0 (좌상)
    3: 1, 4: 1, 7: 1, 8: 1,     # Zone 1 (우상)
    9: 2, 10: 2, 13: 2, 14: 2,  # Zone 2 (좌하)
    11: 3, 12: 3, 15: 3, 16: 3, # Zone 3 (우하)
}
NUM_ZONES = 4

ALL_SUBJECTS = ['gyj', 'jhj', 'jkw', 'kjh', 'kmh', 'kms',
                'kye', 'lsi', 'mhe', 'phr', 'stk', 'swt', 'ysj']

ACTION_NAMES = ['handsup', 'sit', 'stand', 'walk']
ZONE_NAMES = ['Zone0', 'Zone1', 'Zone2', 'Zone3']

# 경로 설정 (실행 환경에 맞게 수정)
PROJECT_ROOT = os.path.abspath(os.path.join(os.getcwd(), '..'))
RAW_DATA_DIR = os.path.join(PROJECT_ROOT, 'RawData')
ALIGNED_DIR = os.path.join(PROJECT_ROOT, 'data_aligned_872')
INTERPOLATED_DIR = os.path.join(PROJECT_ROOT, 'data_interpolated_linear_872')
RESULTS_DIR = os.path.join(os.getcwd(), 'results')
WEIGHTS_DIR = os.path.join(os.getcwd(), 'weights')

os.makedirs(ALIGNED_DIR, exist_ok=True)
os.makedirs(INTERPOLATED_DIR, exist_ok=True)
os.makedirs(RESULTS_DIR, exist_ok=True)
os.makedirs(WEIGHTS_DIR, exist_ok=True)

# 디바이스
def get_device():
    if torch.cuda.is_available():
        return torch.device('cuda')
    return torch.device('cpu')


DEVICE = get_device()
print(f"Device: {DEVICE}")
print(f"Project root: {PROJECT_ROOT}")
print(f"Raw data dir: {RAW_DATA_DIR}")
print(f"Valid HT-LTF subcarriers: {NUM_SUBCARRIERS}")
print(f"Feature dim (114×4 RX): {FEATURE_DIM}")

Device: cpu
Project root: c:\Users\dnxo1\ESP32_YOLO_Motion_Location_Detect
Raw data dir: c:\Users\dnxo1\ESP32_YOLO_Motion_Location_Detect\RawData
Valid HT-LTF subcarriers: 114
Feature dim (114×4 RX): 456


In [9]:
torch.cuda.is_available()

False

---
## Step 1. Raw CSI 데이터 로드 & 진폭 추출

ESP32 CSI 데이터는 각 패킷마다 192개의 서브캐리어에 대한 복소수(I/Q) 값을 포함합니다.

**진폭 추출**: `amplitude = √(Real² + Imaginary²)`

- Raw CSV 구조: 앞 25개 컬럼은 메타데이터, CSI 데이터는 `[R₀ I₀ R₁ I₁ ...]` 형태
- 컬럼 2: Sequence ID (패킷 순번)

In [ ]:
# ============================================================
# 1-1. 진폭 추출 함수
# ============================================================
def extract_amplitude(csi_string):
    """
    Raw CSI 문자열에서 진폭 배열 추출
    Input:  '[R0 I0 R1 I1 R2 I2 ...]' 형태의 문자열
    Output: amplitude 배열 (192,) 또는 (64,) — 서브캐리어 수에 따라 다름
    """
    try:
        s = str(csi_string)
        # CSV 포맷 정리
        for ch in ['"', '[', ']', ',']:
            s = s.replace(ch, ' ')
        vals = np.array([float(v) for v in s.split()])
        if len(vals) < 2 or len(vals) % 2 != 0:
            return None
        real_parts = vals[0::2]  # 짝수 인덱스: Real
        imag_parts = vals[1::2]  # 홀수 인덱스: Imaginary
        amplitude = np.sqrt(real_parts ** 2 + imag_parts ** 2)
        return amplitude
    except Exception:
        return None


# ============================================================
# 1-2. Raw CSV에서 CSI 컬럼 자동 탐색 & 전처리
# ============================================================
def find_csi_column(df):
    """CSI 데이터가 시작되는 컬럼 인덱스 탐색"""
    for col_idx in range(len(df.columns)):
        sample = str(df.iloc[0, col_idx])
        if '[' in sample:
            return col_idx
    return None


def preprocess_raw_csv(filepath):
    """
    Raw CSI CSV → (seq_ids, amplitude_matrix) 반환
    
    Returns:
        seq_ids: (N,) 패킷 순번 배열
        amp_matrix: (N, 114) HT-LTF 서브캐리어 진폭 행렬 (null 제거 완료)
    """
    df = pd.read_csv(filepath, header=None, low_memory=False)
    
    # Sequence ID (컬럼 2)
    seq_ids = df.iloc[:, 2].values.astype(int)
    
    # CSI 컬럼 탐색
    csi_col = find_csi_column(df)
    if csi_col is None:
        raise ValueError(f"CSI column not found in {filepath}")
    
    # 진폭 추출
    amplitudes = []
    valid_seq_ids = []
    for i in range(len(df)):
        amp = extract_amplitude(df.iloc[i, csi_col])
        if amp is None:
            continue
        n_sub = len(amp)
        
        # HT-LTF null subcarrier 제거
        if n_sub == 192:
            amp = amp[VALID_HT_LTF_IDX]  # 192 → 114
        elif n_sub == 64:
            valid_64 = list(range(6, 32)) + list(range(33, 59))
            amp = amp[valid_64]  # 64 → 52
        else:
            continue
        
        amplitudes.append(amp)
        valid_seq_ids.append(seq_ids[i])
    
    amp_matrix = np.array(amplitudes, dtype=np.float32)  # (N, 114)
    valid_seq_ids = np.array(valid_seq_ids)
    return valid_seq_ids, amp_matrix


print("전처리 함수 정의 완료")
print(f"Null subcarrier indices (HT-LTF): {NULL_IN_HT_LTF}")
print(f"Valid subcarrier count: {NUM_SUBCARRIERS}")

In [ ]:
# ============================================================
# 1-3. Raw CSI 데이터 탐색 (샘플 1개 시각화)
# ============================================================
# raw_data 파일 목록 확인
raw_files = sorted(glob.glob(os.path.join(RAW_DATA_DIR, '*_rx1.csv')))
print(f"Raw data 파일 수 (rx1 기준): {len(raw_files)}")
if raw_files:
    print(f"샘플 파일: {os.path.basename(raw_files[0])}")
    
    # 첫 번째 파일로 시각화
    sample_file = raw_files[0]
    seq_ids, amp_matrix = preprocess_raw_csv(sample_file)
    print(f"Seq IDs: {seq_ids[:5]}...{seq_ids[-5:]}")
    print(f"Amplitude matrix shape: {amp_matrix.shape}")  # (N, 114)
    
    fig, axes = plt.subplots(1, 2, figsize=(14, 4))
    # 히트맵
    axes[0].imshow(amp_matrix.T, aspect='auto', cmap='viridis', origin='lower')
    axes[0].set_xlabel('Time (packet index)')
    axes[0].set_ylabel('Subcarrier index')
    axes[0].set_title(f'Raw Amplitude Heatmap\n{os.path.basename(sample_file)}')
    # 특정 서브캐리어 시계열
    for sc in [10, 50, 100]:
        axes[1].plot(amp_matrix[:, sc], label=f'SC {sc}', alpha=0.7)
    axes[1].set_xlabel('Time (packet index)')
    axes[1].set_ylabel('Amplitude')
    axes[1].set_title('Subcarrier Time Series')
    axes[1].legend()
    plt.tight_layout()
    plt.show()
else:
    print("⚠ Raw data 파일이 없습니다. RAW_DATA_DIR 경로를 확인하세요.")

---
## Step 2-3. 872-Frame 정렬 & Null Subcarrier 제거

4개 RX 안테나의 패킷을 `seq_id` 기준으로 동기화(Outer Join)한 후,
초기 50개 패킷을 스킵하고 연속 872프레임을 추출합니다.

- 누락된 패킷은 NaN으로 채워짐 → 다음 단계(보간)에서 처리
- 결과: `data_aligned_872/` 디렉토리에 RX별 CSV 저장

In [ ]:
# ============================================================
# 2-1. 4-RX 동기화 & 872-Frame 정렬 함수
# ============================================================
def align_872_frames(subject, action, position):
    """
    Raw 4-RX CSV → 872-frame 정렬된 CSV 4개 생성
    
    Process:
        1. 4개 RX 파일 로드 → 진폭 추출 + null 서브캐리어 제거
        2. seq_id 기준 Outer Join (동기화)
        3. 초기 50 패킷 스킵, 연속 872 프레임 추출
        4. data_aligned_872/ 에 저장
    
    Returns:
        True if 성공, False if 실패
    """
    rx_data = {}
    all_seq_ids = set()
    
    # 4개 RX 파일 로드
    for rx in range(1, NUM_RX + 1):
        filepath = os.path.join(RAW_DATA_DIR, f"{subject}_{action}_{position}_rx{rx}.csv")
        if not os.path.exists(filepath):
            return False
        seq_ids, amp_matrix = preprocess_raw_csv(filepath)
        if len(seq_ids) == 0:
            return False
        # DataFrame 생성 (seq_id를 인덱스로)
        col_names = [f"sub_{j}" for j in range(amp_matrix.shape[1])]
        rx_df = pd.DataFrame(amp_matrix, index=seq_ids, columns=col_names)
        rx_df = rx_df[~rx_df.index.duplicated(keep='first')]  # 중복 seq_id 제거
        rx_data[rx] = rx_df
        all_seq_ids.update(seq_ids)
    
    # 공통 seq_id 범위 결정
    min_seq = min(all_seq_ids)
    start_seq = min_seq + SKIP_FIRST  # 초기 50 패킷 스킵
    target_indices = list(range(start_seq, start_seq + SEQ_LEN))  # 872개
    
    # 각 RX를 872 프레임에 맞춰 reindex (누락 → NaN)
    for rx in range(1, NUM_RX + 1):
        aligned_df = rx_data[rx].reindex(target_indices)
        out_path = os.path.join(ALIGNED_DIR, f"{subject}_{action}_{position}_rx{rx}_872.csv")
        aligned_df.to_csv(out_path)
    
    return True


# ============================================================
# 2-2. 전체 Raw 데이터 → 872-Frame 정렬 실행
# ============================================================
def run_alignment():
    """13_raw_data/ 전체를 순회하며 872-frame 정렬 수행"""
    # rx1 파일 기준으로 샘플 목록 구성
    rx1_files = sorted(glob.glob(os.path.join(RAW_DATA_DIR, '*_rx1.csv')))
    if not rx1_files:
        print("⚠ Raw data 파일이 없습니다.")
        return 0
    
    success_count = 0
    fail_count = 0
    
    for f in tqdm(rx1_files, desc="872-Frame 정렬"):
        basename = os.path.basename(f).replace('_rx1.csv', '')
        parts = basename.split('_')
        if len(parts) < 3:
            continue
        subject, action, position = parts[0], parts[1], parts[2]
        
        # 이미 정렬된 파일이 있으면 스킵
        out_path = os.path.join(ALIGNED_DIR, f"{subject}_{action}_{position}_rx1_872.csv")
        if os.path.exists(out_path):
            success_count += 1
            continue
        
        ok = align_872_frames(subject, action, position)
        if ok:
            success_count += 1
        else:
            fail_count += 1
    
    print(f"\n정렬 완료: {success_count} 성공, {fail_count} 실패")
    return success_count

# 이미 정렬된 데이터가 있는지 확인
existing_aligned = glob.glob(os.path.join(ALIGNED_DIR, '*_rx1_872.csv'))
if existing_aligned:
    print(f"이미 정렬된 데이터 존재: {len(existing_aligned)}개 샘플")
    print("다시 정렬하려면 run_alignment()를 호출하세요.")
else:
    print("정렬 시작...")
    run_alignment()

---
## Step 4. 선형 보간 (Linear Interpolation)

정렬 과정에서 발생한 NaN (누락 패킷)을 선형 보간으로 채웁니다.
- `interpolate(method='linear')` → `.bfill()` → `.ffill()`
- 결과: `data_interpolated_linear_872/` 에 완전한 (872, 114) CSV 저장

In [ ]:
# ============================================================
# 4-1. 선형 보간 적용
# ============================================================
def run_linear_interpolation():
    """data_aligned_872/ → data_interpolated_linear_872/ 선형 보간"""
    aligned_files = sorted(glob.glob(os.path.join(ALIGNED_DIR, '*_872.csv')))
    if not aligned_files:
        print("⚠ 정렬된 데이터가 없습니다. Step 2-3을 먼저 실행하세요.")
        return 0
    
    count = 0
    for f in tqdm(aligned_files, desc="선형 보간"):
        basename = os.path.basename(f)
        out_path = os.path.join(INTERPOLATED_DIR, basename)
        
        # 이미 보간된 파일이 있으면 스킵
        if os.path.exists(out_path):
            count += 1
            continue
        
        df = pd.read_csv(f, index_col=0)
        nan_count = df.isna().sum().sum()
        
        if nan_count > 0:
            df = df.interpolate(method='linear').bfill().ffill()
        
        df.to_csv(out_path)
        count += 1
    
    print(f"보간 완료: {count}개 파일")
    return count

# 이미 보간된 데이터가 있는지 확인
existing_interp = glob.glob(os.path.join(INTERPOLATED_DIR, '*_rx1_872.csv'))
if existing_interp:
    print(f"이미 보간된 데이터 존재: {len(existing_interp)}개 샘플")
    print("다시 보간하려면 run_linear_interpolation()를 호출하세요.")
else:
    print("선형 보간 시작...")
    run_linear_interpolation()

---
## Step 5. Neural CDE — 불규칙 샘플링 완화 (논문 Section III-A)

논문의 핵심 기여: **Neural Controlled Differential Equations (Neural CDEs)**를 사용하여
불규칙하게 샘플링된 CSI 신호의 연속적 동역학을 모델링합니다.

$$z_t = z_{t_0} + \int_{t_0}^{t} f_\theta(z_s) \frac{dX(s)}{ds} ds \quad \text{(Eq. 2)}$$

- $f_\theta$: 학습 가능한 벡터장 (vector field)
- $X(t)$: 보간된 입력 경로 (natural cubic spline)
- $\ell_\theta(z_t)$: 은닉 상태 → 출력 매핑

**효과**: RNN과 달리 Neural CDE는 동역학적 제약으로 인해 노이즈에 강건한 smooth fitting을 수행합니다 (논문 Fig. 2, 3 참조).

In [ ]:
# ============================================================
# 5-1. Neural CDE 모듈 정의 (논문 Section III-A, Eq. 1-3)
# ============================================================
# torchcde 없이 직접 구현 — torchdiffeq의 ODE solver 활용
#
# pip install torchdiffeq  (필요 시)
# ============================================================
try:
    from torchdiffeq import odeint
    HAS_TORCHDIFFEQ = True
    print("torchdiffeq 로드 성공 — Neural CDE 사용 가능")
except ImportError:
    HAS_TORCHDIFFEQ = False
    print("⚠ torchdiffeq 미설치 — Neural CDE 없이 진행 (선형 보간만 사용)")
    print("  설치: pip install torchdiffeq")


class NaturalCubicSpline:
    """
    Natural Cubic Spline 보간 (논문: X(t)를 differentiable path로 변환)
    
    torchcde 라이브러리의 핵심 기능을 간소화 구현.
    시간 t에서의 보간값과 도함수 dX/dt를 반환합니다.
    """
    def __init__(self, times, values):
        """
        Args:
            times: (L,) 관측 시점
            values: (L, d) 관측값
        """
        self.times = times
        self.values = values
        self.length = times.shape[0]
        self.dim = values.shape[-1]
        
        # 구간별 cubic spline 계수 사전 계산
        self._compute_coefficients()
    
    def _compute_coefficients(self):
        """각 구간의 cubic spline 계수 (a, b, c, d) 계산"""
        t = self.times
        y = self.values
        n = self.length - 1
        
        h = t[1:] - t[:-1]  # (n,) 구간 길이
        
        # Tridiagonal system for natural cubic spline
        # M_0 = M_n = 0 (natural boundary)
        if n < 2:
            # 구간이 1개면 선형 보간
            self.coeffs_a = y[:-1]
            self.coeffs_b = (y[1:] - y[:-1]) / h.unsqueeze(-1)
            self.coeffs_c = torch.zeros_like(y[:-1])
            self.coeffs_d = torch.zeros_like(y[:-1])
            return
        
        # 선형 보간으로 간소화 (대규모 데이터에서 cubic spline의 tridiagonal 풀이 비용 절감)
        # 논문의 핵심은 CDE solver이므로, 입력 경로 X(t)는 선형 보간으로도 충분
        self.coeffs_a = y[:-1]  # (n, d)
        self.coeffs_b = (y[1:] - y[:-1]) / h.unsqueeze(-1)  # (n, d)
    
    def evaluate(self, t_query):
        """
        시점 t_query에서의 보간값 반환
        Args:
            t_query: scalar or (B,) 텐서
        Returns:
            (d,) or (B, d) 보간값
        """
        idx = torch.searchsorted(self.times[:-1], t_query.clamp(self.times[0], self.times[-1])) - 1
        idx = idx.clamp(0, self.length - 2)
        dt = t_query - self.times[idx]
        return self.coeffs_a[idx] + self.coeffs_b[idx] * dt.unsqueeze(-1)
    
    def derivative(self, t_query):
        """
        시점 t_query에서의 도함수 dX/dt 반환 (Eq. 2에서 dX(s)/ds)
        """
        idx = torch.searchsorted(self.times[:-1], t_query.clamp(self.times[0], self.times[-1])) - 1
        idx = idx.clamp(0, self.length - 2)
        return self.coeffs_b[idx]


class CDEFunc(nn.Module):
    """
    Neural CDE의 벡터장 f_θ (논문 Eq. 1-2)
    
    f_θ: R^w → R^(w × (d+1))
    
    CDE 미분방정식: dz/dt = f_θ(z_t) · dX/dt
    이를 ODE 형태로 변환: dz/dt = f_θ(z_t) · dX(t)/dt
    """
    def __init__(self, input_dim, hidden_dim):
        super().__init__()
        self.input_dim = input_dim   # d+1 (features + time)
        self.hidden_dim = hidden_dim  # w
        
        # f_θ: R^w → R^(w × (d+1))
        self.net = nn.Sequential(
            nn.Linear(hidden_dim, hidden_dim * 2),
            nn.Tanh(),
            nn.Linear(hidden_dim * 2, hidden_dim * input_dim),
            nn.Tanh(),
        )
    
    def forward(self, z):
        """
        Args:
            z: (..., w) 은닉 상태
        Returns:
            (..., w, d+1) 벡터장
        """
        out = self.net(z)  # (..., w*(d+1))
        return out.view(*z.shape[:-1], self.hidden_dim, self.input_dim)


class NeuralCDE(nn.Module):
    """
    Neural CDE 모듈 (논문 Section III-A)
    
    Input: irregularly sampled CSI data (B, L, d) with timestamps (B, L)
    Output: CDE-processed CSI data (B, L, d)
    
    Pipeline:
        1. 입력에 시간 차원 추가: (B, L, d) → (B, L, d+1)
        2. Natural Cubic Spline로 연속 경로 X(t) 생성
        3. CDE 풀이: z_t = z_0 + ∫ f_θ(z_s) · dX/ds ds
        4. 출력 매핑: x̂_t = ℓ_θ(z_t)
    """
    def __init__(self, input_dim, hidden_dim=128):
        """
        Args:
            input_dim: CSI feature 차원 (d = 456)
            hidden_dim: CDE 은닉 차원 (w, 논문에서 최적값 400, 여기서는 128 사용)
        """
        super().__init__()
        self.input_dim = input_dim
        self.hidden_dim = hidden_dim
        
        # 초기 은닉 상태: ζ_θ(t_0, x_0) — (논문 Eq. 아래 설명)
        self.initial = nn.Linear(input_dim + 1, hidden_dim)  # d+1 → w
        
        # 벡터장: f_θ
        self.cde_func = CDEFunc(input_dim + 1, hidden_dim)
        
        # 출력 매핑: ℓ_θ: R^w → R^d (논문 Eq. 3)
        self.readout = nn.Linear(hidden_dim, input_dim)
    
    def forward(self, x, timestamps=None):
        """
        Args:
            x: (B, L, d) CSI 데이터
            timestamps: (B, L) or (L,) 관측 시점. None이면 등간격 가정
        Returns:
            x_hat: (B, L, d) CDE 처리된 데이터
        """
        B, L, d = x.shape
        device = x.device
        
        if timestamps is None:
            timestamps = torch.linspace(0, 1, L, device=device)
        if timestamps.dim() == 1:
            timestamps = timestamps.unsqueeze(0).expand(B, -1)
        
        # 시간 차원 추가: (B, L, d) → (B, L, d+1)
        t_feat = timestamps.unsqueeze(-1)  # (B, L, 1)
        x_aug = torch.cat([t_feat, x], dim=-1)  # (B, L, d+1)
        
        outputs = []
        for b in range(B):
            # Spline 경로 생성
            spline = NaturalCubicSpline(timestamps[b], x_aug[b])
            
            # 초기 은닉 상태
            z0 = self.initial(x_aug[b, 0])  # (w,)
            
            # ODE wrapper: dz/dt = f_θ(z) · dX/dt
            cde_func = self.cde_func
            spline_b = spline
            
            def ode_func(t, z, cde_func=cde_func, spline_b=spline_b):
                dXdt = spline_b.derivative(t)  # (d+1,)
                f_z = cde_func(z)              # (w, d+1)
                return (f_z @ dXdt.unsqueeze(-1)).squeeze(-1)  # (w,)
            
            # ODE 풀이
            eval_times = timestamps[b]
            z_trajectory = odeint(ode_func, z0, eval_times, method='euler')  # (L, w)
            
            # 출력 매핑: ℓ_θ(z_t) (Eq. 3)
            x_hat = self.readout(z_trajectory)  # (L, d)
            outputs.append(x_hat)
        
        return torch.stack(outputs, dim=0)  # (B, L, d)


if HAS_TORCHDIFFEQ:
    print("Neural CDE 모듈 정의 완료")
    print(f"  Input dim: {FEATURE_DIM}, Hidden dim: 128")
    print("  Solver: Euler (torchdiffeq.odeint)")
else:
    print("Neural CDE 비활성화 — 선형 보간 데이터로 직접 학습 진행")

---
## Step 6. 데이터셋 구성 & 정규화

보간 완료된 CSV를 PyTorch Dataset으로 변환합니다.
- 4개 RX 파일을 수평 결합: `(872, 114) × 4 → (872, 456)`
- Train 데이터 기준 **StandardScaler** 정규화 (data leakage 방지)
- Dual-task: 각 샘플은 `(x, y_action, y_zone)` 3-tuple 반환

In [ ]:
# ============================================================
# 6-1. CSI Dataset 클래스
# ============================================================
def load_sample(base_dir, subject, action, position):
    """4개 RX CSV를 로드하여 (872, 456) numpy 배열로 반환"""
    rx_list = []
    for rx in range(1, NUM_RX + 1):
        path = os.path.join(base_dir, f"{subject}_{action}_{position}_rx{rx}_872.csv")
        df = pd.read_csv(path, index_col=0)
        rx_list.append(df.values)  # (872, 114)
    return np.hstack(rx_list).astype(np.float32)  # (872, 456)


class CSIDataset(Dataset):
    """
    Wi-Fi CSI 데이터셋 (872 × 456) — Dual-task 전용
    항상 (x, y_action, y_zone) 반환
    """
    def __init__(self, subjects, base_dir=None):
        if base_dir is None:
            base_dir = INTERPOLATED_DIR
        self.base_dir = base_dir
        self.data = []
        self.action_labels = []
        self.zone_labels = []
        self.sample_info = []
        
        rx1_files = glob.glob(os.path.join(base_dir, "*_rx1_872.csv"))
        for f in sorted(rx1_files):
            basename = os.path.basename(f)
            prefix = basename.replace('_rx1_872.csv', '')
            parts = prefix.split('_')
            subject, action, position = parts[0], parts[1], int(parts[2])
            
            if subject not in subjects:
                continue
            if action not in ACTION_MAP:
                continue
            
            try:
                data = load_sample(base_dir, subject, action, str(position))
                if np.isnan(data).any():
                    continue
                self.data.append(data)
                self.action_labels.append(ACTION_MAP[action])
                self.zone_labels.append(ZONE_MAP[position])
                self.sample_info.append((subject, action, position))
            except (FileNotFoundError, Exception):
                continue
        
        self.data = np.array(self.data)  # (N, 872, 456)
        self.action_labels = np.array(self.action_labels)
        self.zone_labels = np.array(self.zone_labels)
        print(f"[CSIDataset] {len(self.data)} samples from {len(subjects)} subjects")
    
    def __len__(self):
        return len(self.data)
    
    def __getitem__(self, idx):
        x = torch.tensor(self.data[idx], dtype=torch.float32)
        y_action = torch.tensor(self.action_labels[idx], dtype=torch.long)
        y_zone = torch.tensor(self.zone_labels[idx], dtype=torch.long)
        return x, y_action, y_zone


def normalize_datasets(train_ds, test_ds):
    """Train 기준 StandardScaler 정규화 (in-place)"""
    scaler = StandardScaler()
    train_flat = train_ds.data.reshape(-1, FEATURE_DIM)
    scaler.fit(train_flat)
    
    train_ds.data = scaler.transform(train_flat).reshape(-1, SEQ_LEN, FEATURE_DIM).astype(np.float32)
    test_flat = test_ds.data.reshape(-1, FEATURE_DIM)
    test_ds.data = scaler.transform(test_flat).reshape(-1, SEQ_LEN, FEATURE_DIM).astype(np.float32)
    return scaler


print("Dataset 클래스 정의 완료")

---
## Step 7. WiDRNet 모델 정의 (논문 Section III-B, III-C)

### Dual-Stream Cross-Attention (Section III-B, Eq. 4, 12-15)
- **Temporal Stream** (Query): `(B, L, d) → (B, L, d_model)` — 시간축 특성 추출
- **Channel Stream** (Key/Value): `(B, d, L) → (B, d, d_model)` — 채널간 상관 추출
- **Cross-Attention**: Q(temporal) × K,V(channel) → 두 스트림의 상관관계 포착
- **Self-Attention + FFN**: 후속 정제 → GAP → 특징 벡터

### Parameter-Shared Dual-Task Learning (Section III-C, Eq. 5-8)
- **Branch 1** (Θ₁): Action Recognition (4 classes)
- **Branch 2** (Θ₂): Zone Classification (4 classes)
- **Regularization**: $\mathcal{L}_{reg} = \lambda \|\Theta_1 - \Theta_2\|^2$ (Eq. 6)
- **Total Loss**: $\mathcal{L} = \mathcal{L}_{action} + \mathcal{L}_{zone} + \mathcal{L}_{reg}$ (Eq. 8)

In [ ]:
# ============================================================
# 7-1. Dual-Stream Cross-Attention 인코더 (논문 Section III-B)
# ============================================================
class DualStreamCrossAttention(nn.Module):
    """
    Temporal stream (Q) × Channel stream (K, V) → Cross-Attention → MHA → FFN → GAP
    
    논문 Eq. 12: Q^CA = X̂ W_q,  K^CA = X̂^T W_k,  V^CA = X̂^T W_v
    논문 Eq. 13-14: Multi-Head Self-Attention on Z^CA
    논문 Eq. 15: FFN with GELU activation
    """
    def __init__(self, input_dim=456, seq_len=872, d_model=128, num_heads=4, d_ffn=256, dropout=0.1):
        super().__init__()
        
        # Temporal Stream: (B, L, d) → (B, L, d_model)
        self.temporal_proj = nn.Linear(input_dim, d_model)
        
        # Channel Stream: (B, d, L) → (B, d, d_model)
        self.channel_proj = nn.Linear(seq_len, d_model)
        
        # Cross-Attention (Eq. 4, 12)
        self.cross_attention = nn.MultiheadAttention(
            embed_dim=d_model, num_heads=num_heads,
            dropout=dropout, batch_first=True
        )
        self.norm_ca = nn.LayerNorm(d_model)
        
        # Multi-Head Self-Attention (Eq. 13-14)
        self.self_attention = nn.MultiheadAttention(
            embed_dim=d_model, num_heads=num_heads,
            dropout=dropout, batch_first=True
        )
        self.norm_mha = nn.LayerNorm(d_model)
        
        # Feed-Forward Network (Eq. 15)
        self.ffn = nn.Sequential(
            nn.Linear(d_model, d_ffn),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(d_ffn, d_model),
            nn.Dropout(dropout),
        )
        self.norm_ffn = nn.LayerNorm(d_model)
    
    def forward(self, x):
        """
        Args:  x: (B, 872, 456)
        Returns: (B, d_model) — GAP 후 특징 벡터
        """
        # Temporal Stream: (B, 872, 456) → (B, 872, 128)
        temporal_q = self.temporal_proj(x)
        
        # Channel Stream: (B, 872, 456) → (B, 456, 872) → (B, 456, 128)
        channel_kv = self.channel_proj(x.transpose(1, 2))
        
        # Cross-Attention (Eq. 12)
        ca_out, _ = self.cross_attention(query=temporal_q, key=channel_kv, value=channel_kv)
        ca_out = self.norm_ca(temporal_q + ca_out)
        
        # Self-Attention (Eq. 13-14)
        mha_out, _ = self.self_attention(query=ca_out, key=ca_out, value=ca_out)
        mha_out = self.norm_mha(ca_out + mha_out)
        
        # FFN (Eq. 15)
        ffn_out = self.ffn(mha_out)
        ffn_out = self.norm_ffn(mha_out + ffn_out)
        
        # Global Average Pooling
        return ffn_out.mean(dim=1)  # (B, 128)


# ============================================================
# 7-2. WiDRNet — Dual-Task Recognition (논문 Section III-C, Fig. 1)
# ============================================================
class WiDRNet(nn.Module):
    """
    두 개의 동일 구조 브랜치(Θ₁, Θ₂)가 각 태스크를 학습하고,
    Parameter Sharing Regularization (Eq. 6)으로 연결됨.
    
    USE_CDE=True일 경우, Neural CDE 전처리 모듈이 인코더 앞에 삽입됨.
    """
    def __init__(self, input_dim=456, seq_len=872, d_model=128, num_heads=4,
                 d_ffn=256, num_actions=4, num_zones=4, fc_hidden=128, dropout=0.1,
                 use_cde=False, cde_hidden=128):
        super().__init__()
        self.use_cde = use_cde
        
        # Neural CDE 전처리 (논문 Section III-A) — Optional
        if use_cde and HAS_TORCHDIFFEQ:
            self.neural_cde = NeuralCDE(input_dim=input_dim, hidden_dim=cde_hidden)
        else:
            self.neural_cde = None
        
        # Branch 1: Action Recognition (Θ₁)
        self.encoder_action = DualStreamCrossAttention(
            input_dim=input_dim, seq_len=seq_len,
            d_model=d_model, num_heads=num_heads, d_ffn=d_ffn, dropout=dropout
        )
        self.classifier_action = nn.Sequential(
            nn.Linear(d_model, fc_hidden), nn.ReLU(), nn.Dropout(dropout),
            nn.Linear(fc_hidden, num_actions),
        )
        
        # Branch 2: Zone Classification (Θ₂)
        self.encoder_zone = DualStreamCrossAttention(
            input_dim=input_dim, seq_len=seq_len,
            d_model=d_model, num_heads=num_heads, d_ffn=d_ffn, dropout=dropout
        )
        self.classifier_zone = nn.Sequential(
            nn.Linear(d_model, fc_hidden), nn.ReLU(), nn.Dropout(dropout),
            nn.Linear(fc_hidden, num_zones),
        )
    
    def forward(self, x):
        """
        Args:  x: (B, 872, 456)
        Returns: (action_logits, zone_logits) — 각각 (B, 4)
        """
        # Neural CDE 전처리 (Optional)
        if self.neural_cde is not None:
            x = self.neural_cde(x)
        
        # Branch 1: Action
        feat_action = self.encoder_action(x)
        action_logits = self.classifier_action(feat_action)
        
        # Branch 2: Zone
        feat_zone = self.encoder_zone(x)
        zone_logits = self.classifier_zone(feat_zone)
        
        return action_logits, zone_logits
    
    def param_reg_loss(self):
        """Parameter Sharing Regularization (Eq. 6): L_reg = ||Θ₁ - Θ₂||²"""
        loss = torch.tensor(0.0, device=next(self.parameters()).device)
        for p1, p2 in zip(self.encoder_action.parameters(), self.encoder_zone.parameters()):
            loss = loss + torch.sum((p1 - p2) ** 2)
        for p1, p2 in zip(self.classifier_action.parameters(), self.classifier_zone.parameters()):
            loss = loss + torch.sum((p1 - p2) ** 2)
        return loss


# 모델 구조 확인
model_test = WiDRNet(use_cde=False).to('cpu')
total_params = sum(p.numel() for p in model_test.parameters())
print(f"WiDRNet (CDE 미적용) 파라미터 수: {total_params:,}")

x_test = torch.randn(2, SEQ_LEN, FEATURE_DIM)
a_logits, z_logits = model_test(x_test)
print(f"Input:  {x_test.shape}")
print(f"Action: {a_logits.shape}, Zone: {z_logits.shape}")
print(f"Reg loss: {model_test.param_reg_loss().item():.4f}")
del model_test, x_test

---
## Step 8. 학습 — Mixup 증강 & Dual-Task Training

### 하이퍼파라미터 (논문 Section IV-B-2 최적값)
| 파라미터 | 값 | 설명 |
|----------|-----|------|
| `d_model` | 128 | 내부 임베딩 차원 |
| `num_heads` | 4 | MHA 헤드 수 (논문 최적) |
| `d_ffn` | 256 | FFN 은닉 차원 (논문 최적) |
| `lambda_reg` | 0.7 | Parameter sharing 강도 (논문 최적) |
| `mixup_alpha` | 0.4 | Beta 분포 파라미터 (논문 최적) |
| `lr` | 1e-3 | Adam optimizer |
| `epochs` | 50 | ~20 에폭 수렴, 여유 50 |

### Mixup (논문 Eq. 17)
$$\tilde{X} = \eta X + (1-\eta) X', \quad \eta \sim \text{Beta}(\alpha, \alpha)$$

### Total Loss (Eq. 8)
$$\mathcal{L} = \mathcal{L}_{action} + \mathcal{L}_{zone} + \lambda \|\Theta_1 - \Theta_2\|^2$$

In [ ]:
# ============================================================
# 8-1. 하이퍼파라미터 설정
# ============================================================
EPOCHS = 50
BATCH_SIZE = 32
LR = 1e-3
WEIGHT_DECAY = 1e-4
LAMBDA_REG = 0.7       # 논문 최적 parameter sharing 강도
MIXUP_ALPHA = 0.4      # 논문 최적 Beta 분포 파라미터
GRAD_CLIP = 1.0
D_MODEL = 128
NUM_HEADS = 4          # 논문 최적
D_FFN = 256            # 논문 최적
DROPOUT = 0.1
USE_CDE = False        # True로 변경 시 Neural CDE 적용 (torchdiffeq 필요)

print(f"Hyperparameters:")
print(f"  epochs={EPOCHS}, batch={BATCH_SIZE}, lr={LR}")
print(f"  lambda_reg={LAMBDA_REG}, mixup_alpha={MIXUP_ALPHA}")
print(f"  d_model={D_MODEL}, heads={NUM_HEADS}, d_ffn={D_FFN}")
print(f"  use_cde={USE_CDE}")

In [ ]:
# ============================================================
# 8-2. Mixup, 평가, Confusion Matrix 유틸리티
# ============================================================
def mixup_data(x, y_action, y_zone, alpha=MIXUP_ALPHA):
    """Mixup (논문 Eq. 17): X̃ = η*X + (1-η)*X', η ~ Beta(α, α)"""
    if alpha > 0:
        lam = np.random.beta(alpha, alpha)
    else:
        lam = 1.0
    index = torch.randperm(x.size(0)).to(x.device)
    mixed_x = lam * x + (1 - lam) * x[index]
    return mixed_x, y_action, y_zone, y_action[index], y_zone[index], lam


def mixup_criterion(criterion, pred, y_a, y_b, lam):
    """Mixup Loss: λ*CE(pred, y_a) + (1-λ)*CE(pred, y_b)"""
    return lam * criterion(pred, y_a) + (1 - lam) * criterion(pred, y_b)


@torch.no_grad()
def evaluate(model, loader, device):
    """모델 평가 — action/zone 정확도 반환"""
    model.eval()
    all_a_pred, all_a_true = [], []
    all_z_pred, all_z_true = [], []
    
    for x, y_act, y_zone in loader:
        x = x.to(device)
        a_logits, z_logits = model(x)
        all_a_pred.append(a_logits.argmax(1).cpu())
        all_a_true.append(y_act)
        all_z_pred.append(z_logits.argmax(1).cpu())
        all_z_true.append(y_zone)
    
    a_pred = torch.cat(all_a_pred).numpy()
    a_true = torch.cat(all_a_true).numpy()
    z_pred = torch.cat(all_z_pred).numpy()
    z_true = torch.cat(all_z_true).numpy()
    
    a_acc = (a_pred == a_true).mean() * 100
    z_acc = (z_pred == z_true).mean() * 100
    return a_acc, z_acc, a_pred, a_true, z_pred, z_true


def plot_confusion_matrix(y_true, y_pred, class_names, title, ax=None):
    """Confusion Matrix 시각화"""
    cm = confusion_matrix(y_true, y_pred)
    if ax is None:
        fig, ax = plt.subplots(figsize=(5, 4))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
                xticklabels=class_names, yticklabels=class_names, ax=ax)
    ax.set_xlabel('Predicted')
    ax.set_ylabel('Actual')
    ax.set_title(title)


print("학습 유틸리티 정의 완료")

In [ ]:
# ============================================================
# 8-3. 학습 루프
# ============================================================
def train_one_split(train_subjects, test_subjects, tag="modeA", save_best=True):
    """
    하나의 train/test split에 대해 학습 및 평가 수행
    
    Returns:
        best_action_acc, best_zone_acc, history dict
    """
    device = DEVICE
    
    # --- 데이터 로드 ---
    print(f"\n{'='*60}")
    print(f"[{tag}] Train: {train_subjects}")
    print(f"[{tag}] Test:  {test_subjects}")
    print(f"{'='*60}")
    
    train_ds = CSIDataset(train_subjects)
    test_ds = CSIDataset(test_subjects)
    
    if len(train_ds) == 0 or len(test_ds) == 0:
        print("데이터 로드 실패!")
        return 0, 0, {}
    
    normalize_datasets(train_ds, test_ds)
    
    train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True, drop_last=False)
    test_loader = DataLoader(test_ds, batch_size=BATCH_SIZE, shuffle=False)
    
    # --- 모델 ---
    model = WiDRNet(
        d_model=D_MODEL, num_heads=NUM_HEADS, d_ffn=D_FFN,
        num_actions=NUM_ACTIONS, num_zones=NUM_ZONES, dropout=DROPOUT,
        use_cde=USE_CDE,
    ).to(device)
    
    criterion = nn.CrossEntropyLoss()
    optimizer = torch.optim.Adam(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
    scheduler = CosineAnnealingLR(optimizer, T_max=EPOCHS)
    
    # 학습 이력 기록
    history = {'loss': [], 'action_acc': [], 'zone_acc': [], 'mean_acc': []}
    best_mean_acc = 0
    best_action_acc = 0
    best_zone_acc = 0
    best_results = None
    
    # --- 학습 루프 ---
    for epoch in tqdm(range(EPOCHS), desc=f"[{tag}] Epochs"):
        model.train()
        total_loss = 0
        num_batches = 0
        
        for x, y_act, y_zone in tqdm(train_loader, desc=f"Epoch {epoch+1}", leave=False):
            x = x.to(device)
            y_act = y_act.to(device)
            y_zone = y_zone.to(device)
            
            # Mixup (Eq. 17)
            mixed_x, y_a_act, y_a_zone, y_b_act, y_b_zone, lam = mixup_data(x, y_act, y_zone)
            
            # Forward
            action_logits, zone_logits = model(mixed_x)
            
            # Total Loss (Eq. 8)
            loss_action = mixup_criterion(criterion, action_logits, y_a_act, y_b_act, lam)
            loss_zone = mixup_criterion(criterion, zone_logits, y_a_zone, y_b_zone, lam)
            loss_reg = model.param_reg_loss()
            loss = loss_action + loss_zone + LAMBDA_REG * loss_reg
            
            # Backward
            optimizer.zero_grad()
            loss.backward()
            clip_grad_norm_(model.parameters(), GRAD_CLIP)
            optimizer.step()
            
            total_loss += loss.item()
            num_batches += 1
        
        scheduler.step()
        
        # Epoch 평가
        avg_loss = total_loss / max(num_batches, 1)
        a_acc, z_acc, a_pred, a_true, z_pred, z_true = evaluate(model, test_loader, device)
        mean_acc = (a_acc + z_acc) / 2
        
        history['loss'].append(avg_loss)
        history['action_acc'].append(a_acc)
        history['zone_acc'].append(z_acc)
        history['mean_acc'].append(mean_acc)
        
        if (epoch + 1) % 5 == 0 or epoch == 0:
            tqdm.write(f"  [{tag}] Epoch {epoch+1:3d} | Loss: {avg_loss:.4f} | "
                       f"Action: {a_acc:.2f}% | Zone: {z_acc:.2f}% | Mean: {mean_acc:.2f}%")
        
        if mean_acc > best_mean_acc:
            best_mean_acc = mean_acc
            best_action_acc = a_acc
            best_zone_acc = z_acc
            best_results = (a_pred, a_true, z_pred, z_true)
            if save_best:
                torch.save(model.state_dict(), os.path.join(WEIGHTS_DIR, f'widr_{tag}_best.pt'))
    
    # --- 최종 결과 ---
    print(f"\n[{tag}] Best Results:")
    print(f"  Action: {best_action_acc:.2f}% | Zone: {best_zone_acc:.2f}% | Mean: {best_mean_acc:.2f}%")
    
    if best_results is not None:
        a_pred, a_true, z_pred, z_true = best_results
        print(f"\n[{tag}] Action Classification Report:")
        print(classification_report(a_true, a_pred, target_names=ACTION_NAMES, zero_division=0))
        print(f"[{tag}] Zone Classification Report:")
        print(classification_report(z_true, z_pred, target_names=ZONE_NAMES, zero_division=0))
    
    return best_action_acc, best_zone_acc, history


print("학습 함수 정의 완료")

### Mode A: Subject-Split (11 train / kjh+kms test)

In [ ]:
# ============================================================
# 8-4. Mode A 실행: 11명 train / kjh+kms 2명 test
# ============================================================
test_subjects_a = ['kjh', 'kms']
train_subjects_a = [s for s in ALL_SUBJECTS if s not in test_subjects_a]

modeA_action, modeA_zone, modeA_history = train_one_split(
    train_subjects_a, test_subjects_a, tag="modeA", save_best=True
)

### Mode B: Subject-wise 5-Fold Cross-Validation

In [ ]:
# ============================================================
# 8-5. Mode B 실행: Subject-wise 5-Fold CV
# ============================================================
FOLDS = [
    ['gyj', 'jhj', 'jkw'],
    ['kjh', 'kmh', 'kms'],
    ['kye', 'lsi', 'mhe'],
    ['phr', 'stk'],
    ['swt', 'ysj'],
]

fold_results = []
fold_histories = []

for i, test_subj in enumerate(FOLDS):
    train_subj = [s for s in ALL_SUBJECTS if s not in test_subj]
    a_acc, z_acc, hist = train_one_split(
        train_subj, test_subj, tag=f"fold{i+1}", save_best=False
    )
    fold_results.append((a_acc, z_acc))
    fold_histories.append(hist)

# --- 5-Fold 결과 요약 ---
print(f"\n{'='*60}")
print("  5-Fold CV Results Summary")
print(f"{'='*60}")
print(f"{'Fold':<8} {'Action(%)':<12} {'Zone(%)':<12} {'Mean(%)':<12}")
print("-" * 44)
for i, (a, z) in enumerate(fold_results):
    print(f"Fold {i+1:<3} {a:<12.2f} {z:<12.2f} {(a+z)/2:<12.2f}")

a_accs = [r[0] for r in fold_results]
z_accs = [r[1] for r in fold_results]
print("-" * 44)
print(f"{'Mean':<8} {np.mean(a_accs):<12.2f} {np.mean(z_accs):<12.2f} {(np.mean(a_accs)+np.mean(z_accs))/2:<12.2f}")
print(f"{'Std':<8} {np.std(a_accs):<12.2f} {np.std(z_accs):<12.2f}")
print(f"\nAction: {np.mean(a_accs):.2f} +/- {np.std(a_accs):.2f}%")
print(f"Zone:   {np.mean(z_accs):.2f} +/- {np.std(z_accs):.2f}%")

---
## Step 9. 결과 시각화

- 학습 곡선 (Loss, Action/Zone Accuracy)
- Confusion Matrix (Action & Zone)
- Mode A vs Mode B 비교 요약

In [ ]:
# ============================================================
# 9-1. Mode A 학습 곡선 시각화
# ============================================================
if modeA_history:
    fig, axes = plt.subplots(1, 3, figsize=(16, 4))
    epochs_range = range(1, len(modeA_history['loss']) + 1)
    
    # Loss
    axes[0].plot(epochs_range, modeA_history['loss'], 'b-')
    axes[0].set_xlabel('Epoch')
    axes[0].set_ylabel('Loss')
    axes[0].set_title('Mode A: Training Loss')
    axes[0].grid(True, alpha=0.3)
    
    # Action Accuracy
    axes[1].plot(epochs_range, modeA_history['action_acc'], 'r-', label='Action')
    axes[1].plot(epochs_range, modeA_history['zone_acc'], 'g-', label='Zone')
    axes[1].set_xlabel('Epoch')
    axes[1].set_ylabel('Accuracy (%)')
    axes[1].set_title('Mode A: Test Accuracy')
    axes[1].legend()
    axes[1].grid(True, alpha=0.3)
    
    # Mean Accuracy
    axes[2].plot(epochs_range, modeA_history['mean_acc'], 'm-')
    axes[2].set_xlabel('Epoch')
    axes[2].set_ylabel('Mean Accuracy (%)')
    axes[2].set_title('Mode A: Mean Dual-task Accuracy')
    axes[2].grid(True, alpha=0.3)
    
    plt.tight_layout()
    plt.savefig(os.path.join(RESULTS_DIR, 'modeA_learning_curve.png'), dpi=150)
    plt.show()
else:
    print("Mode A 학습 이력이 없습니다.")

In [ ]:
# ============================================================
# 9-2. Mode A Confusion Matrix
# ============================================================
# Best 모델을 다시 로드하여 최종 평가
best_weight_path = os.path.join(WEIGHTS_DIR, 'widr_modeA_best.pt')
if os.path.exists(best_weight_path):
    model_eval = WiDRNet(
        d_model=D_MODEL, num_heads=NUM_HEADS, d_ffn=D_FFN,
        num_actions=NUM_ACTIONS, num_zones=NUM_ZONES, dropout=DROPOUT,
        use_cde=USE_CDE,
    ).to(DEVICE)
    model_eval.load_state_dict(torch.load(best_weight_path, map_location=DEVICE, weights_only=True))
    
    test_ds_eval = CSIDataset(test_subjects_a)
    train_ds_eval = CSIDataset(train_subjects_a)
    normalize_datasets(train_ds_eval, test_ds_eval)
    test_loader_eval = DataLoader(test_ds_eval, batch_size=BATCH_SIZE, shuffle=False)
    
    a_acc, z_acc, a_pred, a_true, z_pred, z_true = evaluate(model_eval, test_loader_eval, DEVICE)
    
    fig, axes = plt.subplots(1, 2, figsize=(12, 5))
    plot_confusion_matrix(a_true, a_pred, ACTION_NAMES, 
                         f'Action CM (Mode A)\nAcc: {a_acc:.2f}%', ax=axes[0])
    plot_confusion_matrix(z_true, z_pred, ZONE_NAMES,
                         f'Zone CM (Mode A)\nAcc: {z_acc:.2f}%', ax=axes[1])
    plt.tight_layout()
    plt.savefig(os.path.join(RESULTS_DIR, 'modeA_confusion_matrix.png'), dpi=150)
    plt.show()
    
    del model_eval, test_ds_eval, train_ds_eval
else:
    print("Mode A 최적 모델 가중치가 없습니다.")

In [ ]:
# ============================================================
# 9-3. 5-Fold CV 학습 곡선 비교
# ============================================================
if fold_histories and fold_histories[0]:
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    
    for i, hist in enumerate(fold_histories):
        if hist:
            epochs_range = range(1, len(hist['action_acc']) + 1)
            axes[0].plot(epochs_range, hist['action_acc'], label=f'Fold {i+1}', alpha=0.7)
            axes[1].plot(epochs_range, hist['zone_acc'], label=f'Fold {i+1}', alpha=0.7)
    
    axes[0].set_xlabel('Epoch')
    axes[0].set_ylabel('Accuracy (%)')
    axes[0].set_title('5-Fold CV: Action Recognition')
    axes[0].legend()
    axes[0].grid(True, alpha=0.3)
    
    axes[1].set_xlabel('Epoch')
    axes[1].set_ylabel('Accuracy (%)')
    axes[1].set_title('5-Fold CV: Zone Classification')
    axes[1].legend()
    axes[1].grid(True, alpha=0.3)
    
    plt.tight_layout()
    plt.savefig(os.path.join(RESULTS_DIR, 'modeB_5fold_curves.png'), dpi=150)
    plt.show()
else:
    print("5-Fold CV 학습 이력이 없습니다.")

In [ ]:
# ============================================================
# 9-4. 최종 요약
# ============================================================
print("=" * 70)
print("  WiDR FINAL SUMMARY")
print("=" * 70)

print(f"\nMode A (11 train / kjh+kms test):")
print(f"  Action: {modeA_action:.2f}%")
print(f"  Zone:   {modeA_zone:.2f}%")
print(f"  Mean:   {(modeA_action + modeA_zone) / 2:.2f}%")

if fold_results:
    a_mean = np.mean([r[0] for r in fold_results])
    a_std = np.std([r[0] for r in fold_results])
    z_mean = np.mean([r[1] for r in fold_results])
    z_std = np.std([r[1] for r in fold_results])
    print(f"\nMode B (5-Fold CV):")
    print(f"  Action: {a_mean:.2f} +/- {a_std:.2f}%")
    print(f"  Zone:   {z_mean:.2f} +/- {z_std:.2f}%")
    print(f"  Mean:   {(a_mean + z_mean) / 2:.2f}%")

print(f"\nNeural CDE: {'적용' if USE_CDE else '미적용 (선형 보간만 사용)'}")
print(f"Device: {DEVICE}")
print(f"Total epochs per split: {EPOCHS}")
print("=" * 70)